# Lesson 1: Simple RAG - Complete Implementation


By now, you should have a solid understanding of:

- What embeddings are (geometrically and conceptually)
- Why you can't just give Claude all your documents
- How vector databases work

Now, the task at hand: Use the knowledge you've acquired and build a complete Retrieval-Augmented Generation (RAG) system from scratch.

## What You'll Learn
- Load documents from disk
- Split documents into chunks
- Convert text to embeddings
- Store embeddings in ChromaDB vector database
- Retrieve relevant documents using semantic search
- Generate answers using Claude

## Architecture
```
Load Documents → Chunk Text → Embed Chunks → Store in ChromaDB
                                    ↓
                              User Query
                                    ↓
                        Embed Query + Retrieve
                                    ↓
                          Generate Answer (Claude)
```

Before looking at the `simple_rag_answered.ipynb` complete implementation, try to implement the simple RAG system from scratch.

You can use the following functions and complete them to learn, using everything you learned in the previous lesson:

## 1️⃣ Setup: Import Libraries

In [1]:
from typing import List, Dict
from pathlib import Path
import os

# Vector database
import chromadb

# Embeddings
from sentence_transformers import SentenceTransformer
import numpy as np

# LLM
import anthropic
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("✅ All libraries imported successfully!")

/Users/santiagomora/Documents/personal_projects/learn-agentic-rag-end-to-end/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All libraries imported successfully!


## 2️⃣ Step 1: Create Sample Documents

Let's create some sample documents to load and query.

In [2]:
# Create sample documents directory
sample_docs_dir = Path("sample_documents")
sample_docs_dir.mkdir(exist_ok=True)

# Document 1: Machine Learning Basics
doc1 = """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve 
from experience without being explicitly programmed. It focuses on the development of computer programs 
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning - Learning with labeled data (classification, regression)
2. Unsupervised Learning - Finding patterns in unlabeled data (clustering, dimensionality reduction)
3. Reinforcement Learning - Learning through rewards and penalties (game playing, robotics)

Applications: Image recognition, natural language processing, recommendation systems, autonomous vehicles.
"""

# Document 2: Company Handbook - Leave Policies
doc2 = """Company Handbook - Employee Leave Policies

VACATION LEAVE
All employees receive 15 days of paid vacation per year. Vacation days can be taken at any time 
subject to manager approval. Unused vacation days can carry over up to 5 days into the next year.

PARENTAL LEAVE
New parents are entitled to 16 weeks of paid leave. This applies to both birth parents and non-birthing 
partners. The leave must be taken within 12 months of birth or adoption. During parental leave, 
all benefits including health insurance continue.

SICK LEAVE
Employees receive 10 days of paid sick leave per year. No carryover is allowed. A medical certificate 
is required for absences longer than 3 consecutive days.
"""

# Document 3: Deep Learning
doc3 = """Deep Learning and Neural Networks

Deep learning uses artificial neural networks with multiple layers to model complex patterns in data. 
Each layer learns increasingly abstract representations of the input.

Key Components:
- Neurons: Basic units that perform weighted sum + activation function
- Layers: Input layer, hidden layers, output layer
- Weights and Biases: Learnable parameters optimized during training
- Activation Functions: ReLU, Sigmoid, Tanh enable non-linearity
- Backpropagation: Algorithm to compute gradients and update weights

Popular Architectures: Convolutional Neural Networks (CNNs), Recurrent Neural Networks (RNNs), 
Transformers (BERT, GPT), Vision Transformers (ViT).
"""

# Save documents
with open(sample_docs_dir / "machine_learning.txt", "w") as f:
    f.write(doc1)

with open(sample_docs_dir / "company_handbook.txt", "w") as f:
    f.write(doc2)

with open(sample_docs_dir / "deep_learning.txt", "w") as f:
    f.write(doc3)

print(f"✅ Created {len(list(sample_docs_dir.glob('*.txt')))} sample documents in '{sample_docs_dir}'")

✅ Created 3 sample documents in 'sample_documents'


## 3️⃣ Step 2: Document Loader

Load documents from disk and split them into chunks with metadata.

In [ ]:
class DocumentLoader:
    """Load documents and split them into chunks with metadata."""
    
    def __init__(self, chunk_size: int = 200, chunk_overlap: int = 50):
        """
        Args:
            chunk_size: Number of characters per chunk
            chunk_overlap: Number of characters to overlap between chunks
        """
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
    
    def load_documents(self, directory: str) -> List[Dict]:
        """Load all .txt files from directory.
        
        Returns:
            List of documents with 'content', 'source', and 'path'
        """
        path = Path(directory)
        if not path.exists():
            raise FileNotFoundError(f"Directory not found: {directory}")
        
        documents = []
        for file_path in path.glob("*.txt"):
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
                # TODO: Append a dict with content, source (filename), and path to documents list
                # hints: filepath.name for filename, str(file_path) for path
        
        print(f"📄 Loaded {len(documents)} documents")
        return documents
    
    def chunk_documents(self, documents: List[Dict]) -> List[Dict]:
        """Split documents into chunks with sliding window and overlap.
        
        Returns:
            List of chunks with 'text', 'source', and 'chunk_id'
        """
        chunks = []
        chunk_id = 0
        
        for doc in documents:
            content = doc["content"]
            source = doc["source"]
            
            # Sliding window chunking

            # TODO: Use a loop to create chunks of content with specified chunk_size and chunk_overlap
            # hints: range(0, len(content), chunk_size - chunk_overlap) for loop, content[i:i + chunk_size] for chunk text
            # add a dict with 'text', 'source', and 'chunk_id' to chunks list for each chunk created
        
        print(f"✂️  Split into {len(chunks)} chunks")
        return chunks


# Test the loader

try:
    loader = DocumentLoader(chunk_size=300, chunk_overlap=50)
    documents = loader.load_documents("sample_documents")
    chunks = loader.chunk_documents(documents)

    print(f"\n📊 First chunk preview:")
    print(f"Text: {chunks[0]['text'][:100]}...")
    print(f"Source: {chunks[0]['source']}")
    print(f"Chunk ID: {chunks[0]['chunk_id']}")

    print(f"\n✅ Congratulations! Document loading and chunking successful")

except Exception as e:
    print(f"❌ Error: {e}")
    print(f"Please check the implementation and try again.")

📄 Loaded 3 documents
✂️  Split into 9 chunks

📊 First chunk preview:
Text: Company Handbook - Employee Leave Policies

VACATION LEAVE
All employees receive 15 days of paid vac...
Source: company_handbook.txt
Chunk ID: 0


## 4️⃣ Step 3: Embedder

Convert text chunks into embeddings using SentenceTransformer.

In [4]:
class Embedder:
    """Convert text to embeddings using SentenceTransformer."""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Args:
            model_name: Name of the SentenceTransformer model
                       'all-MiniLM-L6-v2' is small, fast, and good quality
        """
        print(f"🔄 Loading embedding model: {model_name}")
        self.model = SentenceTransformer(model_name)
        self.embedding_dim = self.model.get_sentence_embedding_dimension()
        print(f"✅ Model loaded. Embedding dimension: {self.embedding_dim}")
    
    def embed_text(self, text: str) -> np.ndarray:
        """Embed a single text string.
        
        Args:
            text: Text to embed
        
        Returns:
            Embedding as numpy array of shape (embedding_dim,)
        """
        if not text or not text.strip():
            raise ValueError("Text cannot be empty")
        
        return self.model.encode(text)
    
    def embed_batch(self, texts: List[str], show_progress_bar: bool = True) -> np.ndarray:
        """Embed multiple texts efficiently.
        
        Args:
            texts: List of texts to embed
            show_progress_bar: Show progress during embedding
        
        Returns:
            Embeddings as numpy array of shape (len(texts), embedding_dim)
        """
        if not texts:
            raise ValueError("Texts list cannot be empty")
        
        return self.model.encode(texts, show_progress_bar=show_progress_bar)


# Test the embedder
embedder = Embedder(model_name="all-MiniLM-L6-v2")

# Embed a single text
single_embedding = embedder.embed_text("Hello world")
print(f"\n🔢 Single embedding shape: {single_embedding.shape}")
print(f"First 5 values: {single_embedding[:5]}")

# Embed batch
batch_embeddings = embedder.embed_batch(["Hello", "World"], show_progress_bar=False)
print(f"\n🔢 Batch embeddings shape: {batch_embeddings.shape}")

🔄 Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7024.61it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded. Embedding dimension: 384

🔢 Single embedding shape: (384,)
First 5 values: [-0.03447727  0.03102319  0.00673495  0.02610902 -0.03936202]

🔢 Batch embeddings shape: (2, 384)


## 5️⃣ Step 4: ChromaDB Vector Store

Store documents with embeddings in ChromaDB for semantic search.

In [ ]:
class VectorStore:
    """Manage document storage and retrieval in ChromaDB."""
    
    def __init__(self, persist_dir: str = "./chroma_data"):
        """
        Args:
            persist_dir: Directory to persist ChromaDB data
        """
        self.persist_dir = persist_dir
        self.client = chromadb.PersistentClient(path=persist_dir)
        self.collection = None
        print(f"✅ ChromaDB initialized at {persist_dir}")
    
    def create_collection(self, name: str, clear_if_exists: bool = False) -> None:
        """Create a new collection for storing documents.
        
        Args:
            name: Name of the collection
            clear_if_exists: Delete existing collection with same name
        """
        if clear_if_exists:
            try:
                self.client.delete_collection(name=name)
                print(f"🗑️  Deleted existing collection '{name}'")
            except:
                pass
        # TODO: Create or get collection with the specified name and assign to self.collection
        # hints: self.client.get_or_create_collection(name=name)
        print(f"✅ Created collection '{name}'")
    
    def add_chunks(self, chunks: List[Dict], embeddings: np.ndarray) -> int:
        """Add document chunks with embeddings to the collection.
        
        Args:
            chunks: List of chunks with 'text', 'source', 'chunk_id'
            embeddings: Numpy array of embeddings for each chunk
        
        Returns:
            Number of chunks added
        """
        if not self.collection:
            raise RuntimeError("Create collection first with create_collection()")
        
        if len(chunks) != len(embeddings):
            raise ValueError(f"Mismatch: {len(chunks)} chunks but {len(embeddings)} embeddings")
        
        # Prepare data for ChromaDB
        # TODO: Extract texts, metadatas, and ids from chunks for adding to collection
        # hints: metadatas should include 'source' and 'chunk_id', ids can be generated as f"chunk_{chunk['chunk_id']}"
        texts = []
        metadatas = []
        ids = []
        
        # Add to ChromaDB
        # TODO: Use self.collection.add() to add documents, embeddings, metadatas, and ids to the collection
        
        print(f"💾 Added {len(chunks)} chunks to ChromaDB")
        return len(chunks)
    
    def query(self, query_embedding: np.ndarray, n_results: int = 3) -> List[Dict]:
        """Retrieve similar documents from the collection.
        
        Args:
            query_embedding: Embedding of the query text
            n_results: Number of results to return
        
        Returns:
            List of retrieved documents with text, source, and similarity score
        """
        if not self.collection:
            raise RuntimeError("Create collection first with create_collection()")
        
        # TODO: Use self.collection.query() to retrieve similar documents based on query_embedding
        # hints: query_embeddings should be a list of one embedding, include documents, metadatas, and distances in results
        results = self.collection.query(
            # ...
        )
        
        # Format results
        retrieved = []
        # TODO: Loop through results and extract text, source, and distance to calculate similarity score (similarity = 1 - distance)
        # hints: results has the following fields: 'documents', 'metadatas', 'distances', and you can loop through results["documents"][0] to access each retrieved document and corresponding metadata and distance
        
        return retrieved

try:
    # Initialize and populate vector store
    vector_store = VectorStore(persist_dir="./chroma_data")
    vector_store.create_collection(name="documents", clear_if_exists=True)

    # Embed chunks
    print("\n🔄 Embedding chunks...")
    chunk_texts = [chunk["text"] for chunk in chunks]
    embeddings = embedder.embed_batch(chunk_texts, show_progress_bar=True)

    # Add to vector store
    vector_store.add_chunks(chunks, embeddings)
    print(f"\n✅ Congratulations! Chunks added to vector store successfully")

except Exception as e:
    print(f"❌ Error: {e}")
    print(f"Please check the implementation and try again.")

✅ ChromaDB initialized at ./chroma_data
✅ Created collection 'documents'

🔄 Embedding chunks...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.08it/s]

💾 Added 9 chunks to ChromaDB


9

## 6️⃣ Step 5: Simple RAG System

Orchestrate the entire pipeline: retrieve documents and generate answers with Claude.

In [ ]:
class SimpleRAG:
    """Complete RAG system that retrieves documents and generates answers."""
    
    def __init__(self, embedder: Embedder, vector_store: VectorStore):
        """
        Args:
            embedder: Embedder instance for converting text to embeddings
            vector_store: VectorStore instance for retrieving documents
        """
        self.embedder = embedder
        self.vector_store = vector_store
        self.client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
        print("✅ RAG system initialized")
    
    def retrieve(self, query: str, n_results: int = 3) -> List[Dict]:
        """Retrieve relevant documents for a query.
        
        Args:
            query: User's question
            n_results: Number of documents to retrieve
        
        Returns:
            List of relevant documents
        """
        # Embed the query
        query_embedding = self.embedder.embed_text(query)
        
        # Retrieve from vector store
        results = self.vector_store.query(query_embedding, n_results=n_results)
        
        return results
    
    def answer(self, query: str, n_results: int = 3) -> Dict:
        """Answer a question using retrieved documents and Claude.
        
        Args:
            query: User's question
            n_results: Number of documents to retrieve
        
        Returns:
            Dictionary with 'answer', 'sources', and 'usage'
        """
        # Step 1: Retrieve relevant documents
        print(f"🔍 Retrieving documents for: '{query}'")
        retrieved = self.retrieve(query, n_results=n_results)
        
        # Step 2: Prepare context from retrieved documents
        context = "\n---\n".join([f"From {doc['source']}:\n{doc['text']}" for doc in retrieved])
        sources = list(set([doc["source"] for doc in retrieved]))
        
        print(f"✅ Retrieved {len(retrieved)} chunks from {len(sources)} sources")
        print(f"   Sources: {', '.join(sources)}")
        
        # Step 3: Call Claude with context
        print(f"🤖 Generating answer with Claude...")
        
        # TODO: Use self.client.messages.create() to send a message to Claude with the context and query, and get the answer
        # hints: The message content should include the context and the question.

        message = self.client.messages.create(
            model="claude-haiku-4-5",
            # ...
            )
        
        answer = message.content[0].text
        
        return {
            "answer": answer,
            "sources": sources,
            "retrieved_chunks": len(retrieved),
            "tokens_used": {
                "input": message.usage.input_tokens,
                "output": message.usage.output_tokens
            }
        }


# Initialize RAG system
rag = SimpleRAG(embedder=embedder, vector_store=vector_store)

✅ RAG system initialized


## 7️⃣ Test the RAG System!

Ask questions and get answers powered by Claude and your documents.

In [9]:
# Example 1: Ask about parental leave
print("\n" + "="*70)
print("QUERY 1: How many weeks of parental leave do employees get?")
print("="*70)

result1 = rag.answer("How many weeks of parental leave do employees get?")

print(f"\n📝 ANSWER:\n{result1['answer']}")
print(f"\n📚 Sources: {', '.join(result1['sources'])}")
print(f"📊 Tokens used: {result1['tokens_used']['input']} input, {result1['tokens_used']['output']} output")


QUERY 1: How many weeks of parental leave do employees get?
🔍 Retrieving documents for: 'How many weeks of parental leave do employees get?'
✅ Retrieved 3 chunks from 1 sources
   Sources: company_handbook.txt
🤖 Generating answer with Claude...

📝 ANSWER:
Based on the context provided, employees are entitled to **16 weeks of paid parental leave**. This applies to both birth parents and non-birthing partners, and the leave must be taken within 12 months of birth or adoption. During parental leave, all benefits including health insurance continue.

📚 Sources: company_handbook.txt
📊 Tokens used: 285 input, 65 output


In [10]:
# Example 2: Ask about machine learning
print("\n" + "="*70)
print("QUERY 2: What are the types of machine learning?")
print("="*70)

result2 = rag.answer("What are the three main types of machine learning?")

print(f"\n📝 ANSWER:\n{result2['answer']}")
print(f"\n📚 Sources: {', '.join(result2['sources'])}")
print(f"📊 Tokens used: {result2['tokens_used']['input']} input, {result2['tokens_used']['output']} output")


QUERY 2: What are the types of machine learning?
🔍 Retrieving documents for: 'What are the three main types of machine learning?'
✅ Retrieved 3 chunks from 2 sources
   Sources: deep_learning.txt, machine_learning.txt
🤖 Generating answer with Claude...

📝 ANSWER:
Based on the context provided, the three main types of machine learning are:

1. **Supervised Learning** - Learning with labeled data (classification, regression)
2. **Unsupervised Learning** - Finding patterns in unlabeled data (clustering, dimensionality reduction)
3. **Reinforcement Learning** - Learning through rewards

📚 Sources: deep_learning.txt, machine_learning.txt
📊 Tokens used: 264 input, 76 output


In [11]:
# Example 3: Ask about deep learning
print("\n" + "="*70)
print("QUERY 3: What is backpropagation?")
print("="*70)

result3 = rag.answer("What is backpropagation in deep learning?")

print(f"\n📝 ANSWER:\n{result3['answer']}")
print(f"\n📚 Sources: {', '.join(result3['sources'])}")
print(f"📊 Tokens used: {result3['tokens_used']['input']} input, {result3['tokens_used']['output']} output")


QUERY 3: What is backpropagation?
🔍 Retrieving documents for: 'What is backpropagation in deep learning?'
✅ Retrieved 3 chunks from 1 sources
   Sources: deep_learning.txt
🤖 Generating answer with Claude...

📝 ANSWER:
Based on the context provided, **backpropagation** is an algorithm used in deep learning to compute gradients and update weights.

It is one of the key components that enables neural networks to learn by calculating how much each weight contributed to the error, and then adjusting those weights accordingly during the training process.

📚 Sources: deep_learning.txt
📊 Tokens used: 272 input, 68 output


## 🎯 How It All Works Together

### The Complete Flow:

**Initialization Phase:**
1. **DocumentLoader** → Loads `.txt` files from `sample_documents/` folder
2. **Chunking** → Splits documents into 300-character chunks with 50-character overlap
3. **Embedder** → Converts each chunk to a 384-dimensional vector
4. **VectorStore** → Stores chunks + embeddings + metadata in ChromaDB

**Query Phase:**
1. **User asks a question** → "How many weeks of parental leave?"
2. **Embedder** → Converts question to same 384-dimensional vector
3. **VectorStore.query()** → Finds 3 most similar chunks using distance metric
4. **SimpleRAG** → Sends question + retrieved chunks to Claude
5. **Claude** → Generates answer based on context
6. **Return** → Answer with sources and token usage

### Why This Works:
- **Semantic Search**: Embeddings capture meaning, not just keywords
- **Chunking**: Overlapping chunks preserve context at boundaries
- **Metadata**: We track which document each chunk came from
- **Claude**: Understands and synthesizes information from context

### Key Insights:
✅ **Vector similarity** finds relevant documents automatically  
✅ **Metadata** tracks source documents for attribution  
✅ **Chunking with overlap** preserves context  
✅ **Claude reads** retrieved chunks to generate accurate answers  
✅ **All in one notebook** - easy to understand and modify!

## 🔧 Try Your Own Questions!

Modify the cell below to ask your own questions:

In [12]:
# Try your own question here!
my_question = "What vacation days policy does the company have?"

print("\n" + "="*70)
print(f"QUERY: {my_question}")
print("="*70)

result = rag.answer(my_question)

print(f"\n📝 ANSWER:\n{result['answer']}")
print(f"\n📚 Sources: {', '.join(result['sources'])}")
print(f"📊 Retrieved {result['retrieved_chunks']} chunks")


QUERY: What vacation days policy does the company have?
🔍 Retrieving documents for: 'What vacation days policy does the company have?'
✅ Retrieved 3 chunks from 1 sources
   Sources: company_handbook.txt
🤖 Generating answer with Claude...

📝 ANSWER:
Based on the company handbook, the vacation days policy is as follows:

- **Allocation:** All employees receive 15 days of paid vacation per year
- **Timing:** Vacation days can be taken at any time, subject to manager approval
- **Carryover:** Unused vacation days can carry over up to 5 days into the next year

📚 Sources: company_handbook.txt
📊 Retrieved 3 chunks


## Naive RAG vs Semantic RAG (Hypothetical Document Embeddings (HyDE))

Sometimes, naively using the user's answer as the query to retrieve relevant text chunks can result in retrieving chunks that contain **unneccesary information** that will not provide **relevant context** for the LLM to answer. 

Take this over-exaggerated example:

In [ ]:
my_question = "How many days do the workers of the Machine Learning and Artificial Intelligence team get, given they're regular employees?"

print("\n" + "="*70)
print(f"QUERY: {my_question}")
print("="*70)

result = rag.answer(my_question)

print(f"\n📝 ANSWER:\n{result['answer']}")
print(f"\n📚 Sources: {', '.join(result['sources'])}")
print(f"📊 Retrieved {result['retrieved_chunks']} chunks")


QUERY: How many days do the workers of the Machine Learning and Artificial Intelligence team get, given they're regular employees?
🔍 Retrieving documents for: 'How many days do the workers of the Machine Learning and Artificial Intelligence team get, given they're regular employees?'
✅ Retrieved 3 chunks from 2 sources
   Sources: company_handbook.txt, machine_learning.txt
🤖 Generating answer with Claude...

📝 ANSWER:
Based on the context provided, I cannot answer this question. 

While the company handbook mentions that regular employees receive 10 days of paid sick leave per year, there is no information in the given context about:
- Whether the Machine Learning and Artificial Intelligence team receives different benefits than other regular employees
- Any special leave policies specific to this team
- Total paid leave (vacation days, etc.) beyond sick leave

To answer your question completely, I would need additional information from the company handbook or policies that specifical

In this case, the RAG engine retrieves irrelevant chunks from the `machine_learning.txt` file. 

Some methods aim to **improve the quality of retrieved chunks** by improving the semantic context of what the answer should contain for it to match and retrieve the most relevant information stored in the vector database.

One example is *Hypothetical Document Embeddings (HyDE)*, which passes the original query through a small LLM to write **hypothetical answers** or queries that contain words that would **appear in the answer**. 

In [ ]:
# TODO: Implement HyDE (Hypothetical Document Embeddings) to improve retrieval by generating a 
# semantically optimized hypothetical answer for the query before retrieving documents.
class HydeRAG:
    """Complete RAG system that retrieves documents and generates answers."""
    
    def __init__(self, embedder: Embedder, vector_store: VectorStore):
        """
        Args:
            embedder: Embedder instance for converting text to embeddings
            vector_store: VectorStore instance for retrieving documents
        """
        self.embedder = embedder
        self.vector_store = vector_store
        self.client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
        print("✅ RAG system initialized")
    
    def retrieve(self, query: str, n_results: int = 3) -> List[Dict]:
        """Retrieve relevant documents for a query.
        
        Args:
            query: User's question
            n_results: Number of documents to retrieve
        
        Returns:
            List of relevant documents
        """
        # Embed the query
        query_embedding = self.embedder.embed_text(query)
        
        # Retrieve from vector store
        results = self.vector_store.query(query_embedding, n_results=n_results)
        
        return results
    
    def hyde(self, query: str) -> str:
        """Generate a semantically optimized hypothetical answer (HyDE) for better retrieval.
        
        Args:
            query: User's question
        Returns:
            Hypothetical answer string optimized for retrieval
        """
        # TODO: Use self.client.messages.create() to send a message to Claude asking it to generate a hypothetical answer for the query that would be ideal for retrieval. 
        # The prompt should instruct Claude to create an answer that captures the key concepts and terms related to the query, even if it's not factually correct, to improve retrieval performance.
        hypothetical_answer = ""
                
        return hypothetical_answer.content[0].text
            
    def answer(self, query: str, n_results: int = 3) -> Dict:
        """Answer a question using retrieved documents and Claude.
        
        Args:
            query: User's question
            n_results: Number of documents to retrieve
        
        Returns:
            Dictionary with 'answer', 'sources', and 'usage'
        """
        # Step 0: Generate HyDE query for better retrieval
        print(f"🧠 Generating HyDE query for better retrieval...")
        hyde_query = self.hyde(query)
        print(f"HyDE query: '{hyde_query}'")

        # Step 1: Retrieve relevant documents
        print(f"🔍 Retrieving documents for: '{hyde_query}'")
        retrieved = self.retrieve(hyde_query, n_results=n_results)
        
        # Step 2: Prepare context from retrieved documents
        context = "\n---\n".join([f"From {doc['source']}:\n{doc['text']}" for doc in retrieved])
        sources = list(set([doc["source"] for doc in retrieved]))
        
        print(f"✅ Retrieved {len(retrieved)} chunks from {len(sources)} sources")
        print(f"   Sources: {', '.join(sources)}")
        
        # Step 3: Call Claude with context
        print(f"🤖 Generating answer with Claude...")
        
        message = self.client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=512,
            messages=[
                {
                    "role": "user",
                    "content": f"""Based on the following context, answer the question. 
If the context doesn't contain relevant information, say so.

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""
                }
            ]
        )
        
        answer = message.content[0].text
        
        return {
            "answer": answer,
            "sources": sources,
            "retrieved_chunks": len(retrieved),
            "tokens_used": {
                "input": message.usage.input_tokens,
                "output": message.usage.output_tokens
            }
        }


# Initialize RAG system
hrag = HydeRAG(embedder=embedder, vector_store=vector_store)

✅ RAG system initialized


In [ ]:
my_question = "How many days do the workers of the Machine Learning and Artificial Intelligence team get, given they're regular employees?"

print("\n" + "="*70)
print(f"QUERY: {my_question}")
print("="*70)

result = hrag.answer(my_question)

print(f"\n📝 ANSWER:\n{result['answer']}")
print(f"\n📚 Sources: {', '.join(result['sources'])}")
print(f"📊 Retrieved {result['retrieved_chunks']} chunks")


QUERY: How many days do the workers of the Machine Learning and Artificial Intelligence team get, given they're regular employees?
🧠 Generating HyDE query for better retrieval...
HyDE query: 'Twenty days annual leave yearly.'
🔍 Retrieving documents for: 'Twenty days annual leave yearly.'
✅ Retrieved 3 chunks from 1 sources
   Sources: company_handbook.txt
🤖 Generating answer with Claude...

📝 ANSWER:
Based on the context provided, I can tell you about the leave days for regular employees in general:

- **Vacation Leave**: 15 days per year
- **Sick Leave**: 10 days per year

However, the context does not contain any specific information about whether employees on the Machine Learning and Artificial Intelligence team receive different leave entitlements than regular employees. The handbook excerpts provided only describe the standard leave policies for employees, with no team-specific variations mentioned.

So the answer for regular employees would be **25 days total per year** (15 vaca